# CP6 · Examen blanc de la partie VI — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/checkpoints/partie_6/03_examen_solutions.ipynb)

L'enregistrement des réponses que vérifie le notebook de l'examen (cellules marquées `answer`, récoltées par `tools/build_answers.py`), calculées à partir des nombres du sujet, avec NumPy et des fractions exactes, et recoupées par des vérifications (`assert`) ; les erreurs classiques reçoivent un message qui dit où chercher. Le corrigé détaillé et le barème sont dans `03_examen_corrige.md`.

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

🧪 workbookMaths : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.18 | numpy 2.1.3 | scipy 1.16.3 | sympy 1.14.0 | matplotlib 3.10.0 | pandas 2.2.3 | scikit-learn 1.6.1
   Graine : 42 | FAST_MODE : True


## Les réponses vérifiées après l'examen

In [2]:
# The answers are computed from the numbers of the statement (01_examen_sujet.md), with NumPy and exact fractions
import math
from fractions import Fraction
from itertools import combinations, permutations

import numpy as np


def floats(a):
    """Fractions or floats (a number, nested lists or an array) as plain floats."""
    return np.asarray(a, dtype=float).tolist()


def ints(a):
    """An array (or nested lists) of whole numbers as lists of int, after checking that every value is whole."""
    arr = np.asarray(a, dtype=float)
    assert np.allclose(arr, np.round(arr)), arr
    return np.round(arr).astype(int).tolist()


def fr(x):
    """A number of the statement, written as text ("0.45") or as an int, as an exact fraction."""
    return Fraction(x)


def fmat(rows):
    return [[fr(x) for x in row] for row in rows]


def fmatmul(p, q):
    return [[sum(a * b for a, b in zip(row, col)) for col in zip(*q)] for row in p]


def fmatvec(p, v):
    return [sum(a * b for a, b in zip(row, v)) for row in p]


def fvecmat(v, p):
    """A row vector times a matrix (the row convention of the chains)."""
    return [sum(a * b for a, b in zip(v, col)) for col in zip(*p)]


def ftranspose(p):
    return [list(col) for col in zip(*p)]


def finv2(p):
    """The inverse of a 2 x 2 matrix of fractions."""
    (a, b), (c, d) = p
    det = a * d - b * c
    return [[d / det, -b / det], [-c / det, a / det]]


def fsolve(a, b):
    """The solution x of a x = b, by Gauss-Jordan elimination on fractions."""
    n = len(a)
    m = [list(row) + [b_i] for row, b_i in zip(a, b)]
    for col in range(n):
        pivot = next(r for r in range(col, n) if m[r][col] != 0)
        m[col], m[pivot] = m[pivot], m[col]
        m[col] = [x / m[col][col] for x in m[col]]
        for r in range(n):
            if r != col and m[r][col] != 0:
                m[r] = [x - m[r][col] * y for x, y in zip(m[r], m[col])]
    return [row[-1] for row in m]


def fsqrt(q):
    """The exact square root of a fraction that is a perfect square (25 -> 5)."""
    q = Fraction(q)
    num, den = math.isqrt(q.numerator), math.isqrt(q.denominator)
    assert num * num == q.numerator and den * den == q.denominator, q
    return Fraction(num, den)


def as_typed(value, right, places=(4, 3, 2), decimals=6):
    """A wrong value as a learner may write it where the statement asks for an exact value: exact, then rounded to 4,
    3 and 2 decimals, without the roundings that are the same value at the recorded decimals, or the right answer.
    A set for a number (wbm gives one message to all), a list of variants for a list (see same_message)."""
    def key(v):
        return tuple(np.round(np.ravel(np.asarray(v, dtype=float)), decimals))

    seen, variants = {key(right)}, []
    for v in [value] + [np.round(np.asarray(value, dtype=float), k) for k in places]:
        v = floats(v)
        if key(v) not in seen:
            seen.add(key(v))
            variants.append(v)
    return set(variants) if np.ndim(value) == 0 else variants


def same_message(message, values):
    """One message for several wrong values of a list or of a fraction (wbm takes one value per message there): the
    keys differ by trailing spaces, which the message shown leaves out."""
    return {message + " " * i: v for i, v in enumerate(values)}


# CP6.1: the birth mass x1 and the mass at one month x2 of a calf (kg)
MU_1, S_1 = [fr(40), fr(55)], fmat([[16, 12], [12, 13]])
X1_1, X2_1, EPS_1, M_1 = fr(48), fr("48.5"), [fr(1), fr("-2.5")], fmat([[0, 4], [2, 3]])
L_1 = np.linalg.cholesky(np.array(floats(S_1)))
assert np.allclose(L_1 @ L_1.T, floats(S_1)) and L_1[0, 1] == 0 and (np.diag(L_1) > 0).all()
L11_1 = math.sqrt(S_1[0][0])
L21_1 = float(S_1[1][0]) / L11_1
assert np.allclose(L_1, [[L11_1, 0], [L21_1, math.sqrt(S_1[1][1] - L21_1 ** 2)]])   # the formulas of M25a
L_ROOT_1 = [[L11_1, 0], [L21_1, math.sqrt(S_1[1][1])]]                 # l22 = √Σ22: the l21² forgotten
L21_SIGMA_1 = float(S_1[1][0] / S_1[0][0])                             # Σ21 divided by Σ11, not by l11
L_SIGMA_1 = [[L11_1, 0], [L21_SIGMA_1, math.sqrt(S_1[1][1] - L21_SIGMA_1 ** 2)]]


def conditional(read, other, value, s=S_1, mu=MU_1):
    """The law of x_other given x_read = value, [mean, variance] (the Schur complement), exactly."""
    gain = s[other][read] / s[read][read]
    return [mu[other] + gain * (value - mu[read]), s[other][other] - gain * s[read][other]]


MARG_1 = [MU_1[1], S_1[1][1]]
C21_1 = conditional(0, 1, X1_1)
C12_1 = conditional(1, 0, X2_1)
assert math.isclose(C21_1[1], L_1[1, 1] ** 2)                          # var(x2 | x1) = l22²
MARG_COND_1 = [MU_1[1], C21_1[1]]                                     # the variance read given x1
MARG_SD_1 = [float(MU_1[1]), math.sqrt(S_1[1][1])]
C21_OTHER_1 = [MU_1[1] + S_1[1][0] / S_1[1][1] * (X1_1 - MU_1[0]), S_1[1][1] - S_1[1][0] ** 2 / S_1[1][1]]
C21_MEAN_1 = [MU_1[1] + S_1[1][0] / S_1[0][0] * (X1_1 - MU_1[1]), C21_1[1]]   # the gap read from the other mean
C12_OTHER_1 = [MU_1[0] + S_1[0][1] / S_1[0][0] * (X2_1 - MU_1[1]), S_1[0][0] - S_1[0][1] ** 2 / S_1[0][0]]
C12_SIGN_1 = [MU_1[0] - S_1[0][1] / S_1[1][1] * (X2_1 - MU_1[1]), C12_1[1]]
DRAW_1 = (np.array(floats(MU_1)) + L_1 @ floats(EPS_1)).tolist()
DRAW_T_1 = (np.array(floats(MU_1)) + L_1.T @ floats(EPS_1)).tolist()   # Lᵀ in place of L
DRAW_S_1 = floats([m + e for m, e in zip(MU_1, fmatvec(S_1, EPS_1))])  # Σ in place of L
DRAW_0_1 = (L_1 @ floats(EPS_1)).tolist()                              # the mean forgotten
COV_M_1 = fmatmul(M_1, ftranspose(M_1))
RNG_1 = np.random.default_rng(6801)
E_1 = RNG_1.standard_normal((200_000, 2))
assert np.allclose(np.cov((E_1 @ np.array(floats(M_1)).T).T), floats(COV_M_1), atol=0.2)   # the draws M ε

# CP6.2 (1): two pollutants, the true law p0 and the model p1
MU0_2, S0_2 = [fr(1), fr(0)], fmat([[5, 3], [3, 4]])
MU1_2, S1_2 = [fr(3), fr(1)], fmat([[4, 2], [2, 2]])
D_2 = [b - a for a, b in zip(MU0_2, MU1_2)]


def det2(s):
    return s[0][0] * s[1][1] - s[0][1] * s[1][0]


def trace(s):
    return sum(s[i][i] for i in range(len(s)))


def kl_terms(m0, s0, m1, s1):
    """The three terms of KL(N(m0, s0) ‖ N(m1, s1)) in dimension 2: tr(s1⁻¹ s0), δᵀ s1⁻¹ δ and ln(det s1 / det s0)."""
    inv = finv2(s1)
    delta = [b - a for a, b in zip(m0, m1)]
    return trace(fmatmul(inv, s0)), sum(d * x for d, x in zip(delta, fmatvec(inv, delta))), math.log(det2(s1) / det2(s0))


def kl_gauss(m0, s0, m1, s1):
    tr, q, ln = kl_terms(m0, s0, m1, s1)
    return 0.5 * (float(tr) + float(q) - len(m0) + ln)


TR_2, Q_2, LN_2 = kl_terms(MU0_2, S0_2, MU1_2, S1_2)
KL_2 = kl_gauss(MU0_2, S0_2, MU1_2, S1_2)


def log_density(x, m, s):
    s = np.array(floats(s))
    d = x - np.array(floats(m))
    return -0.5 * np.einsum("ni,ij,nj->n", d, np.linalg.inv(s), d) - 0.5 * math.log(np.linalg.det(2 * math.pi * s))


RNG_2 = np.random.default_rng(6802)
SAMPLE_2 = RNG_2.multivariate_normal(floats(MU0_2), floats(S0_2), size=200_000)
assert abs(np.mean(log_density(SAMPLE_2, MU0_2, S0_2) - log_density(SAMPLE_2, MU1_2, S1_2)) - KL_2) < 0.01   # E_p0
TR_SWAP_2 = kl_terms(MU0_2, S1_2, MU1_2, S0_2)[0]                    # tr(Σ0⁻¹Σ1)
TR_COF_2 = TR_2 * det2(S1_2)                                       # the cofactors not divided by the determinant
TR_PROD_2 = trace(finv2(S1_2)) * trace(S0_2)
Q_DIRECT_2 = sum(d * x for d, x in zip(D_2, fmatvec(S1_2, D_2)))   # Σ1 in place of its inverse
Q_OTHER_2 = sum(d * x for d, x in zip(D_2, fmatvec(finv2(S0_2), D_2)))
Q_PLAIN_2 = sum(d * d for d in D_2)
Q_DIAG_2 = sum(d * d / s for d, s in zip(D_2, (S1_2[0][0], S1_2[1][1])))   # Σ1 without its off-diagonal terms
KL_LOG_2 = 0.5 * (float(TR_2) + float(Q_2) - 2 - LN_2)             # ln(det Σ0 / det Σ1)
KL_NO_D_2 = 0.5 * (float(TR_2) + float(Q_2) + LN_2)                # the dimension not subtracted
KL_NO_HALF_2 = float(TR_2) + float(Q_2) - 2 + LN_2
KL_REVERSE_2 = kl_gauss(MU1_2, S1_2, MU0_2, S0_2)

# CP6.2 (2): u and v, of the same norm, and the slerp at t = 1/4
U_2, V_2, T_2 = np.array([2.0, 2, 1]), np.array([0.0, 3, 0]), 0.25
NORM_2 = float(np.linalg.norm(U_2))
assert math.isclose(NORM_2, np.linalg.norm(V_2))
OMEGA_2 = math.acos(U_2 @ V_2 / NORM_2 ** 2)
DEGREES_2 = math.degrees(OMEGA_2)


def slerp(u, v, t, omega=OMEGA_2):
    return (math.sin((1 - t) * omega) * u + math.sin(t * omega) * v) / math.sin(omega)


SLERP_2 = slerp(U_2, V_2, T_2)
assert math.isclose(np.linalg.norm(SLERP_2), NORM_2) and math.isclose(math.acos(SLERP_2 @ U_2 / NORM_2 ** 2), T_2 * OMEGA_2)
ASIN_2 = math.degrees(math.asin(U_2 @ V_2 / NORM_2 ** 2))
SWAPPED_2 = slerp(V_2, U_2, T_2)                                   # the two coefficients exchanged
LINE_2 = (1 - T_2) * U_2 + T_2 * V_2
PROJECTED_2 = NORM_2 * LINE_2 / np.linalg.norm(LINE_2)            # the straight point brought back to the sphere
LINE_NORM_2 = float(np.linalg.norm(LINE_2))
MIDDLE_2 = float(np.linalg.norm((U_2 + V_2) / 2))

# CP6.3: delivery times (h), two warehouses, a known common variance, one EM step on the weight and the means
PI_3, MU_3, S2_3, X_3 = np.array([0.62, 0.38]), np.array([24.0, 32.0]), 16.0, np.array([22.0, 27.0, 36.0])


def weighted_densities(pi, mu=MU_3, s2=S2_3, x=X_3, half=True):
    """π_k N(x_i | μ_k, σ²), one row per measure (half=False: the square of the gap divided by σ² only)."""
    gaps = (x[:, None] - mu[None, :]) ** 2 / (2 * s2 if half else s2)
    return pi[None, :] * np.exp(-gaps) / math.sqrt(2 * math.pi * s2)


def responsibilities(pi, half=True):
    w = weighted_densities(pi, half=half)
    return w / w.sum(axis=1, keepdims=True)


R_3 = responsibilities(PI_3)
LOGIT_3 = np.log(PI_3[0] / PI_3[1]) + ((X_3 - MU_3[1]) ** 2 - (X_3 - MU_3[0]) ** 2) / (2 * S2_3)
assert np.allclose(R_3[:, 0], 1 / (1 + np.exp(-LOGIT_3)))          # a sigmoid of the log-odds
N_3 = R_3.sum(axis=0)
PI_NEW_3 = float(N_3[0] / len(X_3))
MU_NEW_3 = (R_3 * X_3[:, None]).sum(axis=0) / N_3
HARD_3 = R_3.argmax(axis=1)                                        # the component of largest responsibility
LOG_P_3 = np.log(weighted_densities(PI_3).sum(axis=1)).sum()
ELBO_HARD_3 = np.log(weighted_densities(PI_3)[np.arange(len(X_3)), HARD_3]).sum()   # H(q) = 0 for a sure q
GAP_3 = float(LOG_P_3 - ELBO_HARD_3)
assert math.isclose(GAP_3, -np.log(R_3[np.arange(len(X_3)), HARD_3]).sum())   # a sum of KL(sure ‖ posterior) = −ln r
R_EQUAL_3 = responsibilities(np.array([0.5, 0.5]))
R_NO_HALF_3 = responsibilities(PI_3, half=False)
MU_BY_N_3 = (R_3 * X_3[:, None]).sum(axis=0) / len(X_3)
GAP_ALL1_3 = float(-np.log(R_3[:, 0]).sum())                       # every measure put in component 1
ENTROPY_3 = float(-(R_3 * np.log(R_3)).sum())
HARD_SHARE_3 = float(np.mean(HARD_3 == 0))                         # the share of the measures put in component 1

# CP6.4: vine plots, young (0), producing (1), old (2); the mass missing from a row is the end
P_4 = fmat([["0.6", "0.33", 0], [0, "0.7", "0.2"], [0, 0, "0.4"]])
R_4 = [fr(0), fr(22), fr(6)]
START_4 = [fr(1), fr(0), fr(0)]
MU1_4 = fvecmat(START_4, P_4)
MU2_4 = fvecmat(MU1_4, P_4)
MU3_4 = fvecmat(MU2_4, P_4)
P2_4 = fmatmul(P_4, P_4)
FULL_4 = np.zeros((4, 4))                                          # the end as an absorbing fourth state
FULL_4[:3, :3] = floats(P_4)
FULL_4[:3, 3] = 1 - FULL_4[:3, :3].sum(axis=1)
FULL_4[3, 3] = 1
OUT2_4 = 1 - sum(MU2_4)
assert np.allclose(np.linalg.matrix_power(FULL_4, 2)[0], floats(MU2_4 + [OUT2_4]))
I_MINUS_4 = [[int(i == j) - P_4[i][j] for j in range(3)] for i in range(3)]
V_4 = fsolve(I_MINUS_4, R_4)
assert all(P_4[i][j] == 0 for i in range(3) for j in range(i))     # triangular: solved from V(2) up
assert np.allclose(sum(np.linalg.matrix_power(np.array(floats(P_4)), k) @ floats(R_4) for k in range(500)), floats(V_4))
YEAR2_4 = sum(m * r for m, r in zip(MU2_4, R_4))
COLUMN_4 = [row[0] for row in P2_4]                               # P² μ0ᵀ: the law put on the right of P
SQUARES_4 = [p ** 2 for p in P_4[0]]
OUT_FIRST_4, OUT_SECOND_4 = 1 - sum(MU1_4), sum(MU1_4) - sum(MU2_4)
V_NO_LOOP_4 = [fr(0)] * 3
for s in (2, 1, 0):                                                # the diagonal coefficient of each row forgotten
    V_NO_LOOP_4[s] = R_4[s] + sum(P_4[s][j] * V_NO_LOOP_4[j] for j in range(3) if j != s)
V_T_4 = fsolve(ftranspose(I_MINUS_4), R_4)
V_FUTURE_4 = [v - r for v, r in zip(V_4, R_4)]
YEARS_4 = {sum(m * r for m, r in zip(MU1_4, R_4)), sum(m * r for m, r in zip(MU3_4, R_4))}   # years 1 and 3
RENORM_4 = YEAR2_4 / sum(MU2_4)

# CP6.5 (1): the poacher (rows: north, south) and the warden (columns: north, south)
G_5 = [[-4, 6], [3, -2]]
MAXIMIN_5 = max(min(row) for row in G_5)
MINIMAX_5 = min(max(col) for col in zip(*G_5))
assert MAXIMIN_5 < MINIMAX_5                                       # no saddle point: mixed strategies
(A_5, B_5), (C_5, D_5) = G_5
KAPPA_5 = A_5 - B_5 - C_5 + D_5
P_5, Q_5, VALUE_5 = Fraction(D_5 - C_5, KAPPA_5), Fraction(D_5 - B_5, KAPPA_5), Fraction(A_5 * D_5 - B_5 * C_5, KAPPA_5)
assert P_5 * A_5 + (1 - P_5) * C_5 == P_5 * B_5 + (1 - P_5) * D_5 == VALUE_5   # the warden is indifferent
assert Q_5 * A_5 + (1 - Q_5) * B_5 == Q_5 * C_5 + (1 - Q_5) * D_5 == VALUE_5   # the poacher is indifferent
GRID_5 = [Fraction(k, 600) for k in range(601)]                    # brute force on a grid of mixed strategies
assert max(GRID_5, key=lambda p: min(p * A_5 + (1 - p) * C_5, p * B_5 + (1 - p) * D_5)) == P_5
assert min(GRID_5, key=lambda q: max(q * A_5 + (1 - q) * B_5, q * C_5 + (1 - q) * D_5)) == Q_5
COLUMN_MIN_5 = max(min(col) for col in zip(*G_5))                  # the warden's worst read as a minimum

# CP6.5 (2): the heights (cm) of five plants of each nursery
XA_5, XB_5 = [26, 19, 34, 21, 32], [29, 37, 14, 22, 18]
GAPS_5 = [abs(a - b) for a, b in zip(sorted(XA_5), sorted(XB_5))]
W1_5 = Fraction(sum(GAPS_5), len(GAPS_5))
W2_5 = math.sqrt(sum(g * g for g in GAPS_5) / len(GAPS_5))
PAIRINGS_5 = list(permutations(XB_5))                              # every pairing: the sorted one is optimal
assert W1_5 == min(Fraction(sum(abs(a - b) for a, b in zip(XA_5, p)), 5) for p in PAIRINGS_5)
assert math.isclose(W2_5, min(math.sqrt(sum((a - b) ** 2 for a, b in zip(XA_5, p)) / 5) for p in PAIRINGS_5))
UNSORTED_W1_5 = Fraction(sum(abs(a - b) for a, b in zip(XA_5, XB_5)), 5)
MEANS_5 = abs(Fraction(sum(XA_5), 5) - Fraction(sum(XB_5), 5))
UNSORTED_W2_5 = math.sqrt(sum((a - b) ** 2 for a, b in zip(XA_5, XB_5)) / 5)

# CP6.6: three buoys, the squares of their distances (hm²)
AB2_6, AC2_6, BC2_6 = 16, 121, 49
D2_6 = fmat([[0, AB2_6, AC2_6], [AB2_6, 0, BC2_6], [AC2_6, BC2_6, 0]])
N_6 = len(D2_6)
J_6 = [[int(i == j) - Fraction(1, N_6) for j in range(N_6)] for i in range(N_6)]
S_6 = Fraction(sum(map(sum, D2_6)), N_6 ** 2)
JDJ_6 = fmatmul(fmatmul(J_6, D2_6), J_6)
G_6 = [[-x / 2 for x in row] for row in JDJ_6]
ROWS_6 = [sum(row) / N_6 for row in D2_6]
assert G_6 == [[-(D2_6[i][j] - ROWS_6[i] - ROWS_6[j] + S_6) / 2 for j in range(N_6)] for i in range(N_6)]
assert all(sum(row) == 0 for row in G_6)                           # G 1 = 0
assert all(G_6[i][j] * G_6[k][l] == G_6[i][l] * G_6[k][j]          # every 2 x 2 minor is zero: G has rank 1
           for i, k in combinations(range(N_6), 2) for j, l in combinations(range(N_6), 2))
TRACE_6 = trace(G_6)                                               # rank 1: the only non-zero eigenvalue
LAMBDAS_6 = [TRACE_6] + [Fraction(0)] * (N_6 - 1)
assert np.allclose(sorted(np.linalg.eigvalsh(np.array(floats(G_6))), reverse=True), floats(LAMBDAS_6), atol=1e-9)
W_6, Q_6 = np.linalg.eigh(np.array(floats(G_6)))
COLUMNS_6 = []
for k in np.argsort(-W_6):
    if W_6[k] > 1e-9:                                              # one column per non-zero eigenvalue
        unit = Q_6[:, k] * np.sign(Q_6[np.flatnonzero(np.round(Q_6[:, k], 12))[0], k])   # first non-zero coordinate > 0
        COLUMNS_6.append(math.sqrt(W_6[k]) * unit)
MAP_6 = np.column_stack(COLUMNS_6)
V_6 = [x / fsqrt(G_6[0][0]) for x in (row[0] for row in G_6)]      # G = v vᵀ, v read on the first column
assert fmatmul([[x] for x in V_6], [V_6]) == G_6 and np.allclose(MAP_6[:, 0], floats(V_6))
assert np.allclose(((MAP_6[:, None, :] - MAP_6[None, :, :]) ** 2).sum(axis=-1), floats(D2_6))   # the distances back
OFF_DIAGONAL_6 = Fraction(sum(map(sum, D2_6)), N_6 * (N_6 - 1))
ONCE_6 = Fraction(AB2_6 + AC2_6 + BC2_6, N_6 ** 2)
SIGN_S_6 = [[-(D2_6[i][j] - ROWS_6[i] - ROWS_6[j] - S_6) / 2 for j in range(N_6)] for i in range(N_6)]
J_FLOAT_6 = np.array(floats(J_6))
DIST_6 = -0.5 * J_FLOAT_6 @ np.sqrt(np.array(floats(D2_6))) @ J_FLOAT_6   # the distances, not their squares
NONZERO_6 = [x for x in LAMBDAS_6 if x != 0]                       # the zero eigenvalues left out
DIAGONAL_6 = [G_6[i][i] for i in range(N_6)]                       # the diagonal read as eigenvalues
LAMBDA_Q_6 = MAP_6 * np.sqrt(W_6[W_6 > 1e-9][::-1])[None, :]      # λ q in place of √λ q
UNIT_6 = MAP_6 / np.sqrt(W_6[W_6 > 1e-9][::-1])[None, :]           # q alone

# CP6.7: two solar farms, daily production of two rows of panels (MWh)
MUA_7, SA_7 = np.array([18.0, 12.0]), np.array([[12.0, 2.0], [2.0, 9.0]])
MUB_7, SB_7 = np.array([21.0, 13.0]), 4 * SA_7
NA_7, NB_7, NE_7 = 20, 40, 25
LAM_AB_7 = sorted(np.linalg.eigvals(SA_7 @ SB_7).real, reverse=True)
EIG_A_7 = sorted(np.linalg.eigvalsh(SA_7), reverse=True)
assert np.allclose(LAM_AB_7, [4 * x ** 2 for x in EIG_A_7])       # Σ_B = 4 Σ_A
ROOTS_7 = [math.sqrt(x) for x in LAM_AB_7]
COV_7 = float(np.trace(SA_7) + np.trace(SB_7) - 2 * sum(ROOTS_7))


def sqrtm_sym(s):
    w, q = np.linalg.eigh(s)
    return q @ np.diag(np.sqrt(w)) @ q.T


ROOT_A_7 = sqrtm_sym(SA_7)
assert math.isclose(np.trace(SA_7 + SB_7 - 2 * sqrtm_sym(ROOT_A_7 @ SB_7 @ ROOT_A_7)), COV_7)
MEANS_7 = float(((MUA_7 - MUB_7) ** 2).sum())
W2_7 = math.sqrt(MEANS_7 + COV_7)
TRA_7, TRB_7 = float(np.trace(SA_7)), float(np.trace(SB_7))


def bias(n_a, n_b):
    """E‖x̄A − x̄B‖² for n_a days of A and n_b days of B: the mean of n days is N(μ, Σ/n)."""
    return MEANS_7 + TRA_7 / n_a + TRB_7 / n_b


BIAS_7, SAME_7 = bias(NA_7, NB_7), bias(NE_7, NE_7)
RNG_7 = np.random.default_rng(6803)
for n_a, n_b in ((NA_7, NB_7), (NE_7, NE_7)):                      # Monte Carlo: the means of n days
    bars_a = RNG_7.multivariate_normal(MUA_7, SA_7 / n_a, size=20_000)
    bars_b = RNG_7.multivariate_normal(MUB_7, SB_7 / n_b, size=20_000)
    assert abs(((bars_a - bars_b) ** 2).sum(axis=1).mean() - bias(n_a, n_b)) < 0.15


def pooled(n_a, n_b):
    """The sum of the traces divided by the total number of days, in place of tr ΣA / nA + tr ΣB / nB."""
    return MEANS_7 + (TRA_7 + TRB_7) / (n_a + n_b)


COV_ONCE_7 = float(TRA_7 + TRB_7 - sum(ROOTS_7))
COV_NO_ROOT_7 = float(TRA_7 + TRB_7 - 2 * sum(LAM_AB_7))
TRACES_7 = TRA_7 + TRB_7
NO_N_7 = MEANS_7 + TRA_7 + TRB_7
SWAP_N_7 = MEANS_7 + TRA_7 / NB_7 + TRB_7 / NA_7
NO_MEANS_7 = TRA_7 / NA_7 + TRB_7 / NB_7
GAP_NORM_7 = math.sqrt(math.sqrt(MEANS_7) + COV_7)                 # the distance of the means, not its square
SAME_A_7 = 2 * TRA_7 / NE_7                                        # two samples of farm A
ONE_TERM_7 = {MEANS_7 + TRA_7 / NE_7, MEANS_7 + TRB_7 / NE_7}      # one estimated mean only
NO_MEANS_E_7 = (TRA_7 + TRB_7) / NE_7

# CP6.9 a): the letters of the five wrong lines
LINES_9, WRONG_9 = "ABCDEFGH", "ACEGH"
RIGHT_9 = "".join(sorted(set(LINES_9) - set(WRONG_9)))
FOUR_9 = {"".join(c) for c in combinations(LINES_9, 4)}           # the messages depend on the number of letters only
SIX_9 = {"".join(c) for c in combinations(LINES_9, 6)}
FIVE_9 = {"".join(c) for c in combinations(LINES_9, 5)} - {WRONG_9}
assert (len(FOUR_9), len(SIX_9), len(FIVE_9), RIGHT_9) == (70, 28, 55, "BDF")

# CP6.10: a charging station with two waiting places, 0, 1 or 2 cars waiting, every five minutes
P_10 = fmat([["0.32", "0.60", "0.08"], ["0.21", "0.33", "0.46"], ["0.05", "0.60", "0.35"]])
M_10, U0_10 = fr("0.68"), 0
assert all(sum(row) == 1 for row in P_10)
GREEDY_10 = [max(range(3), key=lambda j: (row[j], -j)) for row in P_10]   # ties: the smallest state
COLUMN_10 = [max(range(3), key=lambda i: (P_10[i][j], -i)) for j in range(3)]
SEQ_10 = [U0_10]
while SEQ_10[-1] not in SEQ_10[:-1]:
    SEQ_10.append(GREEDY_10[SEQ_10[-1]])
PRE_10 = SEQ_10.index(SEQ_10[-1])
PERIOD_10 = len(SEQ_10) - 1 - PRE_10
LONG_10 = [U0_10]
for _ in range(30):
    LONG_10.append(GREEDY_10[LONG_10[-1]])
assert all(LONG_10[t] == LONG_10[t + PERIOD_10] for t in range(PRE_10, 25))
assert all(LONG_10[t] not in LONG_10[PRE_10:PRE_10 + PERIOD_10] for t in range(PRE_10))


def top_p(row, m=M_10):
    """The states kept, by decreasing probability (ties: the smallest state first), until their mass reaches m."""
    kept = []
    for j in sorted(range(len(row)), key=lambda j: (-row[j], j)):
        kept.append(j)
        if sum(row[k] for k in kept) >= m:
            return kept


KEPT_10 = [top_p(row) for row in P_10]
for row, kept in zip(P_10, KEPT_10):                               # the smallest set whose mass reaches m
    assert len(kept) == min(len(c) for k in range(1, 4) for c in combinations(range(3), k)
                            if sum(row[j] for j in c) >= M_10)
P_TRUNC_10 = [[row[j] / sum(row[k] for k in kept) if j in kept else fr(0) for j in range(3)]
              for row, kept in zip(P_10, KEPT_10)]
ROW1_10, KEPT1_10, PS1_10 = P_10[1], KEPT_10[1], P_TRUNC_10[1]
MASS1_10 = sum(ROW1_10[j] for j in KEPT1_10)
KL1_10 = sum(float(q) * math.log(q / p) for q, p in zip(PS1_10, ROW1_10) if q > 0)
assert math.isclose(KL1_10, -math.log(MASS1_10))                   # the cost of a truncation: −ln P(S)


def stationary(p):
    """π P = π and Σ π = 1, exactly: two equations of (Pᵀ − I) π = 0 and the sum."""
    rows = [[p[j][i] - int(i == j) for j in range(3)] for i in range(3)][:2] + [[fr(1)] * 3]
    pi = fsolve(rows, [fr(0), fr(0), fr(1)])
    assert fvecmat(pi, p) == pi
    return pi


PI_NEW_10 = stationary(P_TRUNC_10)
assert np.allclose(np.ones(3) / 3 @ np.linalg.matrix_power(np.array(floats(P_TRUNC_10)), 400), floats(PI_NEW_10))
PI_OLD_10 = stationary(P_10)
ORDER1_10 = sorted(range(3), key=lambda j: (-ROW1_10[j], j))
LONGER_10 = ORDER1_10[:len(KEPT1_10) + 1]                          # one state more than needed
SHORTER_10 = ORDER1_10[:len(KEPT1_10) - 1]                         # one state less: the mass is under m
assert sum(ROW1_10[j] for j in SHORTER_10) < M_10
KEPT_ONLY_10 = [[row[j] if j in kept else fr(0) for j in range(3)] for row, kept in zip(P_10, KEPT_10)]
SPREAD_10 = [ROW1_10[j] + (1 - MASS1_10) / len(KEPT1_10) if j in KEPT1_10 else fr(0) for j in range(3)]
ONE_HOT_10 = [[int(j == GREEDY_10[i]) for j in range(3)] for i in range(3)]
UNNORMALISED_10 = [x / PI_NEW_10[1] for x in PI_NEW_10]           # π′ scaled so that π′1 = 1, not to sum 1
SWAPPED_10 = [PI_NEW_10[0], PI_NEW_10[2], PI_NEW_10[1]]

**CP6.1 — Un vecteur gaussien de dimension 2 : loi marginale, loi conditionnelle, et la loi de μ + Lε**

In [3]:
wbm.record("CP6.1a", L_1.tolist(), decimals=6, mistakes={**same_message('L est triangulaire inférieure : ses coefficients au-dessus de la diagonale sont nuls', as_typed(L_1.T, L_1)), **same_message('le coefficient l₂₂ est la racine de Σ₂₂ − l₂₁², pas de Σ₂₂', as_typed(L_ROOT_1, L_1)), **same_message('le coefficient l₂₁ vaut Σ₂₁ / l₁₁ : divise par l₁₁, pas par Σ₁₁', as_typed(L_SIGMA_1, L_1))})
wbm.record("CP6.1b", floats(MARG_1), decimals=6, mistakes={**same_message("une loi marginale ne tient compte d'aucune valeur lue", as_typed(MARG_COND_1, MARG_1)), **same_message('le second paramètre de N(μ, σ²) est une variance, pas un écart-type', as_typed(MARG_SD_1, MARG_1))})
wbm.record("CP6.1c", floats(C21_1), decimals=6, mistakes={**same_message("lire x₁ réduit l'incertitude sur x₂ : la variance conditionnelle est le complément de Schur, plus petit que Σ₂₂", as_typed([C21_1[0], S_1[1][1]], C21_1)), **same_message('dans la loi de x₂ sachant x₁, on divise par la variance de la coordonnée lue', as_typed(C21_OTHER_1, C21_1)), **same_message("l'écart lu se mesure à la moyenne de la coordonnée lue", as_typed(C21_MEAN_1, C21_1))})
wbm.record("CP6.1d", floats(C12_1), decimals=4, mistakes={"lire x₂ réduit l'incertitude sur x₁ : la variance conditionnelle est le complément de Schur, plus petit que Σ₁₁": floats([C12_1[0], S_1[0][0]]), 'dans la loi de x₁ sachant x₂, on divise par la variance de la coordonnée lue': floats(C12_OTHER_1), "vérifie le signe de l'écart lu, x₂ − μ₂": floats(C12_SIGN_1), 'dans la variance conditionnelle de x₁, on divise par la variance de la coordonnée lue, Σ₂₂': floats([C12_1[0], C12_OTHER_1[1]])})
wbm.record("CP6.1e", DRAW_1, decimals=6, mistakes={**same_message('le tirage est μ + Lε, avec L, pas Lᵀ', as_typed(DRAW_T_1, DRAW_1)), **same_message('le tirage multiplie ε par le facteur de Cholesky, pas par Σ', as_typed(DRAW_S_1, DRAW_1)), **same_message('le tirage part de la moyenne', as_typed(DRAW_0_1, DRAW_1))})
wbm.record("CP6.1f", floats(COV_M_1), decimals=6, mistakes={**same_message('la covariance de Mε est M Mᵀ, pas Mᵀ M', as_typed(fmatmul(ftranspose(M_1), M_1), COV_M_1)), **same_message("la covariance d'une image linéaire Mε est M I Mᵀ : il faut une transposée", as_typed(fmatmul(M_1, M_1), COV_M_1))})

WBM_ANSWER {"decimals": 6, "element_coarse": [["0c934e6e0a0b9da5060c8010a5390ae70ce81e6ac80c1638d07eb1e6f29ceaca"], ["0689a8ff0ea87d307e5532d9e2c246b51089005647f89dc2d371f81148fcc85b"], ["e3b6997cf0301a78c86c599f336f43421f42f37455029079fd928c13d21f7138"], ["23e5e80f0bb79535a9599dd95f34d730bb4061de9ea1e2dbba7e380e482d0057"]], "element_coarse_any": [["2eee57a17ee568ac8d86fe9aa3feb1e09224bb2107c0b6d422c32bb4c22aa6b6", "ce033bc3e1b2ee5e3980d9d7ef97e2853009a94120da9b2e3dd891faafd38897", "6e49df3fdd104b9f88ea0f13fe62e6d08f05ebe09c2b078d3fc338fb848aae5f", "57dfc5940efc88d3aae481333f76e7058ac63a0acfa83ffc888bd177845c479c", "23f27e33b36ecf6240919e28152c359a5c43aeb24613895294f5bfd272550442"], [], ["0b7ada644eaeb341ee39a624e28f6b8b8c7fe293114166b81a3cd87f5e29ac95", "4f01dfb8dd69c116656001cfb69602962dfe3ce68ffb690f28bfc77d3239ae51", "02c744731949bda05cd2f888b91a1e850e3b082c93c8a4885563a5e0e19d90f7", "dc97cfe892cdee628510e1a70f7e5870eecb2181efe629fc0ce8bc198504823b", "20f96214b5db5b40c6cc67c09ce682

**CP6.2 — La divergence KL entre deux gaussiennes de dimension 2, et une interpolation sphérique**

In [4]:
wbm.record("CP6.2a", float(TR_2), decimals=6, mistakes={"la trace de la formule est celle de Σ₁⁻¹Σ₀ : l'inverse de la covariance de la seconde loi": as_typed(TR_SWAP_2, TR_2), "l'inverse d'une matrice 2 × 2 est sa comatrice transposée, divisée par son déterminant": as_typed(TR_COF_2, TR_2), "la trace d'un produit n'est pas le produit des traces": as_typed(TR_PROD_2, TR_2)})
wbm.record("CP6.2b", float(Q_2), decimals=6, mistakes={"le terme des moyennes est une distance de Mahalanobis : il utilise l'inverse d'une covariance": as_typed(Q_DIRECT_2, Q_2), "le terme des moyennes se mesure à l'échelle de la seconde loi, p₁": as_typed(Q_OTHER_2, Q_2), 'la distance de Mahalanobis tient compte des covariances': as_typed(Q_PLAIN_2, Q_2), 'Σ₁ a des coefficients hors diagonale : inverse la matrice entière': as_typed(Q_DIAG_2, Q_2)})
wbm.record("CP6.2c", KL_2, decimals=4, mistakes={"le logarithme compare les déterminants dans l'ordre det Σ₁ / det Σ₀": KL_LOG_2, 'la formule retranche la dimension d': KL_NO_D_2, 'la formule a un facteur ½ devant le crochet': KL_NO_HALF_2, "la KL demandée est KL(p₀ ‖ p₁) : l'espérance se prend sous la première loi": KL_REVERSE_2})
wbm.record("CP6.2d", DEGREES_2, decimals=2, mistakes={'la réponse est attendue en degrés, pas en radians': OMEGA_2, "l'angle Ω a pour cosinus uᵀv / (‖u‖ ‖v‖)": ASIN_2})
wbm.record("CP6.2e", SLERP_2.tolist(), decimals=4, mistakes={'le coefficient de u est sin((1 − t)Ω)/sin Ω, celui de v est sin(tΩ)/sin Ω': SWAPPED_2.tolist(), "le slerp suit l'arc de grand cercle, pas la ligne droite": LINE_2.tolist(), "ramener sur la sphère le point de la ligne droite ne donne pas l'angle tΩ": PROJECTED_2.tolist()})
wbm.record("CP6.2f", LINE_NORM_2, decimals=3, mistakes={'la norme est la racine de la somme des carrés': LINE_NORM_2 ** 2, "le point de la ligne droite n'est pas sur la sphère : calcule sa norme": NORM_2, 'la question porte sur le point t = 1/4 de la ligne droite, pas sur son milieu': MIDDLE_2})

WBM_ANSWER {"coarse_any": {"0": ["5e996f9e52eaa6b692c9296f154ef968ab7168d7d2371d41b999b1cf5907bc9b"]}, "coarser": {"1": ["597e58e93de3fd2de90b5459b979acff1c7112450383371d269323b0bf1aa68d"], "2": ["051fa451b577153d16dbdd7a7a7dfcb09f0577795ed321cef23bbdc3d2be7dad"], "3": ["0e61ffc7390979f00598fc1de26e8cd2aa645e047c00b2c73dc68582baf69679"], "4": ["13a7341a09764c9b882f70b141f181c2f054b95e120a10a249c077d989d32617"]}, "decimals": 6, "hash": "871fa8e0c284ef32211ba3b756c5f34162289fd1d150ad21e46e22d2229caf10", "hash_coarse": "8e8286c441d6c91b1773146c8a7738ffe7bff82bdd2a5bd59958759a706806ac", "id": "CP6.2a", "kind": "float", "magnitude": 0, "mistakes": {"19d363d697c309cf66f9992ca08317708d3e323b8c41632b780d29f5989bda1c": "la trace de la formule est celle de Σ₁⁻¹Σ₀ : l'inverse de la covariance de la seconde loi", "1ccf2c631088eb97ae4cdcb163012316eabf21b808bc8dc5b49c7e5f2d925713": "la trace de la formule est celle de Σ₁⁻¹Σ₀ : l'inverse de la covariance de la seconde loi", "3351f4f92e86a4d0763dead77

**CP6.3 — Un mélange de deux gaussiennes de variances connues et trois mesures : responsabilités, un pas d'EM sur le poids et les moyennes, et l'écart à l'ELBO d'une affectation dure**

In [5]:
wbm.record("CP6.3a", R_3[:, 0].tolist(), decimals=4, mistakes={'les responsabilités tiennent compte des poids πₖ': R_EQUAL_3[:, 0].tolist(), 'la responsabilité demandée est celle de la composante 1, de moyenne 24 h': R_3[:, 1].tolist(), "dans la densité gaussienne, le carré de l'écart est divisé par 2σ²": R_NO_HALF_3[:, 0].tolist()})
wbm.record("CP6.3b", PI_NEW_3, decimals=4, mistakes={'un poids est une proportion : les deux poids ont pour somme 1': float(N_3[0]), "le pas M remplace l'ancien poids : recalcule-le à partir des responsabilités": float(PI_3[0]), 'π₁ est la moyenne des responsabilités, pas la part des graines affectées à la composante 1': HARD_SHARE_3})
wbm.record("CP6.3c", MU_NEW_3.tolist(), decimals=3, mistakes={'une moyenne pondérée se divise par la somme de ses poids': MU_BY_N_3.tolist(), "les moyennes se donnent dans l'ordre des composantes (24 h et 32 h au départ)": MU_NEW_3[::-1].tolist()})
wbm.record("CP6.3d", GAP_3, decimals=4, mistakes={"l'affectation dure met chaque mesure dans la composante de plus grande responsabilité": GAP_ALL1_3, 'un écart à une borne inférieure est positif ou nul': -GAP_3, "l'écart à l'ELBO est une divergence KL, pas une entropie": ENTROPY_3})

WBM_ANSWER {"decimals": 4, "element_coarse": [["be2cd1225bfec1ac7209501b2bfd4ebd0670507061aaf45a9a93c4f263fe3af8"], ["a3261581158289309724d45bd055d1b24901f427a1b3c68fdb302075a6932cb6"], ["507c665b5beada0f4f48d256d2b4b35e5c8c0e02e723bd89fe4609a907cfdc20"]], "element_coarse_any": [["0c8961142e1b67af969ede40f7e090ef78a08a56725c5f567b3c4e52b36ccce6", "50a26c5f8ee2dc1794670aa592dc08882c2ff15980f7baf14fcd7958f7174465", "329786856dfe6e89597d731432820bebd9516c7e030826c1017f88b882bff022"], ["70b20e646be980af2ceb95136cad3594e661ffe4c1d736d567ebdd56426ca1bd", "b444f2497d095cc5c75693af4aa9d07ea8a8ab82fb7fbda2f0a17189e56ac61f", "a603d0d71ecab1e0db198c9a7e475f525fbc9ce645aabdd7b5475e94da9b9591"], ["f7f5c1ff659309c52532a23e13451d0f2d561e07334eee7f11e12f5ef2ad3745"]], "element_hashes": ["6b0ec67e6b3277f4df4a4da7dff6df2660db817da634dbc1c3c5af1bd915d3cd", "c199503fbb569a0dab00b9593227d36c33e3472f8104b4f141bdd63c3cd2583b", "2e89e843700c09ce65aa4a5f40c10818a097a207b5edce4d4a5283b0b3a792c3"], "hash": "a551

**CP6.4 — Une chaîne de Markov à trois états : loi à deux pas, et la valeur d'une politique fixée par le système de Bellman, quand une ligne finit**

In [6]:
wbm.record("CP6.4a", floats(MU2_4), decimals=6, mistakes={**same_message('en convention ligne, une loi avance par μP : elle est à gauche de P', as_typed(COLUMN_4, MU2_4)), **same_message("un coefficient de P² se lit sur les chemins de longueur 2 : ce n'est pas le carré d'un coefficient de P", as_typed(SQUARES_4, MU2_4)), **same_message('la question demande la loi dans deux ans, après deux pas', as_typed(MU1_4, MU2_4))})
wbm.record("CP6.4b", float(OUT2_4), decimals=6, mistakes={'la parcelle peut sortir du domaine la première année ou la deuxième : compte les deux': as_typed(OUT_FIRST_4, OUT2_4) | as_typed(OUT_SECOND_4, OUT2_4), "la question demande la probabilité d'être sortie du domaine, pas d'y être encore": as_typed(sum(MU2_4), OUT2_4), "la deuxième année, la parcelle n'est plus forcément jeune : la probabilité de sortir cette année-là se calcule avec la loi μ₁, pas avec la seule ligne 0": as_typed(2 * OUT_FIRST_4, OUT2_4)})
wbm.record("CP6.4c", floats(V_4), decimals=6, mistakes={**same_message('une parcelle peut rester dans le même état : la ligne de P garde son coefficient diagonal', as_typed(V_NO_LOOP_4, V_4)), **same_message('en convention ligne, V = r + PV : la ligne s de P donne les états suivants de s', as_typed(V_T_4, V_4)), **same_message("la valeur compte aussi la récolte de l'année en cours", as_typed(V_FUTURE_4, V_4)), "la fin n'est pas un état de P : la liste donne V(0), V(1) et V(2)": floats(V_4 + [0])})
wbm.record("CP6.4d", float(YEAR2_4), decimals=6, mistakes={'l\'année 0 est l\'année en cours : la troisième récolte est celle de l\'année 2': set().union(*(as_typed(y, YEAR2_4) for y in YEARS_4)), 'une parcelle sortie du domaine ne récolte plus rien : ne renormalise pas la loi': as_typed(RENORM_4, YEAR2_4)})

WBM_ANSWER {"array_rounded": {"1": "7348d8faec658edada14ba2afb6710b6c7415db6a0620747b77e80992b440e66", "2": "bc4d431c293066e5cd0bcb38b6e8b4ebaaef8eeabd93130404c801b3c3c35700"}, "decimals": 6, "element_coarse": [["98be0d7f4b1571bdfd0f1bfe8d5d92d9ed74e7b6bd0dc15e81dd58aa83f991c3"], ["516213e5e6108781d2bbf5429758926b18087b1d0145634cfc1ece0fa97ddcbb"], ["93a82158e2a4ce7e6ac7750fc15ca08278cb86c33ccbde417ffca6973bb2201a"]], "element_coarse_any": [["523207722fc3f95d69e60a29c94b6ab20e2ef63e485113ec82a11413380781c9", "272bbd72a4395b78fc1549d8f72f0aef1c344ec11794d8d3de1e1dedc943faec", "af4e306a83df1a40f5b52218cf5ac0338cd8a7865e9d67e66e36bc69fa818791", "1077cf3bfb8ec2c1c468e78e5bdb6654201ce5d29bc1b31185f4543c7c1b5da2"], ["618cd6f596a31254a44057246bb1ca887bc0d3590ff21137361df2714f8e4cd5", "d391644c7d82ec55d9e5c431490efb3700c530db4acce3be7a77d875571dad2a", "610d8e658967faadd4ef144c542c24a596951bd704cadb0a83c558063b696212", "d0b422b3a5275cdb9c83c1f345e28c82456024d1e3ec3ea88543d4757918c49d"], ["6a1f6

**CP6.5 — Un jeu à somme nulle 2 × 2 en stratégies mixtes, et W₁ entre deux petits échantillons de même taille**

In [7]:
wbm.record("CP6.5a", [MAXIMIN_5, MINIMAX_5], mistakes={'le joueur des colonnes paie : son pire, dans chaque colonne, est le plus grand gain du braconnier': [MAXIMIN_5, COLUMN_MIN_5], "le maximin (ce que s'assure le braconnier) vient en premier, et il est au plus le minimax": [MINIMAX_5, MAXIMIN_5]})
wbm.record("CP6.5b", str(P_5), kind="fraction", simplified=True, mistakes={"la stratégie optimale du braconnier rend le garde indifférent : écris l'égalité des deux colonnes": str(Q_5), 'la probabilité p est celle de la première ligne, le nord': str(1 - P_5)})
wbm.record("CP6.5c", str(Q_5), kind="fraction", simplified=True, mistakes={"la stratégie optimale du garde rend le braconnier indifférent : écris l'égalité des deux lignes": str(P_5), 'la probabilité q est celle de la première colonne, le nord': str(1 - Q_5)})
wbm.record("CP6.5d", str(VALUE_5), kind="fraction", simplified=True, mistakes={**same_message('sans point selle, la valeur en stratégies mixtes est strictement entre le maximin et le minimax', [MAXIMIN_5, MINIMAX_5]), 'vérifie le signe de κ = a − b − c + d': str(-VALUE_5)})
wbm.record("CP6.5e", float(W1_5), decimals=6, mistakes={"trie les deux échantillons avant d'apparier": as_typed(UNSORTED_W1_5, W1_5), "l'écart des moyennes n'est qu'une minoration de W₁": as_typed(MEANS_5, W1_5), 'la distance W₁ est la moyenne des écarts appariés, pas leur somme': as_typed(sum(GAPS_5), W1_5)})
wbm.record("CP6.5f", W2_5, decimals=4, mistakes={'la distance W₂ est la racine de la moyenne des carrés des écarts': W2_5 ** 2, "trie les deux échantillons avant d'apparier": UNSORTED_W2_5})

WBM_ANSWER {"decimals": 0, "element_hashes": ["6323302df3e89785319b02bfdc71503239da2380e99807f8f9f19d0ce6f21623", "59c3464de87a8de88ed4c1089eabf91b09ca9492b7d7d2e7be1ebab841e206f2"], "hash": "e74b737dec93095598fee84290c5c3a2d7f91abe0471e01c41861ca98a9fd6de", "id": "CP6.5a", "integer": true, "kind": "array", "mistakes": {"56e1efd1946c72b91a258f4026b5cd8f696f08ac8c5985c98bfb0628aa9b5bd9": "le maximin (ce que s'assure le braconnier) vient en premier, et il est au plus le minimax", "9a97c2a9b6f1e6ee0dec4310d1c6d84caab6b757a3ddb6fc22b870b27023af5f": "le joueur des colonnes paie : son pire, dans chaque colonne, est le plus grand gain du braconnier"}, "shape": [2]}
📝 Ex CP6.5a : réponse enregistrée (array, 0 décimale).


WBM_ANSWER {"hash": "07c7a338bdcc43ea47451287af087d5a84eec05447078d7239cd0d0b93c299fa", "id": "CP6.5b", "kind": "fraction", "mistakes": {"1880a4ae49fd2e899c3bec4fe74d39201725502987623ef660bd3e918cf01ccd": "la probabilité p est celle de la première ligne, le nord", "6f1a615086c145cc09aeb70f16d0c9484bd9fbd870e2e43fcf48fead9e7fad8d": "la stratégie optimale du braconnier rend le garde indifférent : écris l'égalité des deux colonnes"}, "sign": 1, "simplified": true}
📝 Ex CP6.5b : réponse enregistrée (fraction).
WBM_ANSWER {"hash": "d29d5eb859d87c044edd63b0ebd2f0e64ac6bd43e0baf562700f4cafd19e0913", "id": "CP6.5c", "kind": "fraction", "mistakes": {"2ca40bdca116699b5f4f75b9102a506bb15659390c4c95aed19510a1d39d0fb8": "la stratégie optimale du garde rend le braconnier indifférent : écris l'égalité des deux lignes", "c009a732bdb48f4b91495930f9f20a65dc1ba364cb171fabf51885211544f0a6": "la probabilité q est celle de la première colonne, le nord"}, "sign": 1, "simplified": true}
📝 Ex CP6.5c : réponse 

**CP6.6 — Des distances aux positions : double centrage, matrice de Gram, et une configuration à une isométrie près**

In [8]:
wbm.record("CP6.6a", str(S_6), kind="fraction", simplified=True, mistakes={'la moyenne porte sur les neuf coefficients de D⁽²⁾, les zéros de la diagonale compris': str(OFF_DIAGONAL_6), 'chaque carré de distance apparaît deux fois dans D⁽²⁾': str(ONCE_6)})
wbm.record("CP6.6b", floats(G_6), decimals=6, mistakes={**same_message("la matrice de Gram est −½ J D⁽²⁾ J : n'oublie pas le facteur −½", as_typed(JDJ_6, G_6)), **same_message('dans Gᵢⱼ = −½(D⁽²⁾ᵢⱼ − rᵢ − rⱼ + s), vérifie le signe de s', as_typed(SIGN_S_6, G_6)), **same_message('le double centrage se fait sur les carrés des distances', as_typed(DIST_6, G_6))})
wbm.record("CP6.6c", floats(LAMBDAS_6), decimals=6, mistakes={**same_message('les valeurs propres se donnent de la plus grande à la plus petite', as_typed(LAMBDAS_6[::-1], LAMBDAS_6)), 'G a trois valeurs propres : donne aussi les nulles': floats(NONZERO_6), **same_message("les coefficients diagonaux ne sont les valeurs propres que d'une matrice diagonale", as_typed(DIAGONAL_6, LAMBDAS_6))})
wbm.record("CP6.6d", MAP_6.tolist(), decimals=6, mistakes={**same_message('dans chaque colonne, la première coordonnée (celle de A) est positive', as_typed(-MAP_6, MAP_6)), **same_message('la colonne k est √λₖ qₖ, avec qₖ unitaire', as_typed(LAMBDA_Q_6, MAP_6)), **same_message('une colonne de la carte est le vecteur propre unitaire multiplié par la racine de sa valeur propre', as_typed(UNIT_6, MAP_6))})

WBM_ANSWER {"hash": "25da3613b2e642df7ace420e023bc1ae2c6858c5789ffe3399f5e8fd23596369", "id": "CP6.6a", "kind": "fraction", "mistakes": {"b3566f49a5db3a2cc0a3899f3841d68143a97210127cdbc987cdc73cba854f8d": "la moyenne porte sur les neuf coefficients de D⁽²⁾, les zéros de la diagonale compris", "fd79cb896473a2c565e297de4160cdbd9e637b5ee76c42e79363aa88a342a1bc": "chaque carré de distance apparaît deux fois dans D⁽²⁾"}, "sign": 1, "simplified": true}
📝 Ex CP6.6a : réponse enregistrée (fraction).
WBM_ANSWER {"decimals": 6, "element_coarse": [["245657318c1245350b0d70870532d4fff04077996438b62cabc6465df99a4cf3"], ["a48963822ce3902a0ca4c43446956799b4e8bff8ac95aa01c97043c77bb3038a"], ["49defde936fb24b04d9972d5356dd0eb6bf84f62c1565460f5696a58f8f31940"], ["e4a010b98da9cf75bf06de8c49d0dfc255bc077fdab7dccf61af1c5d8eed81bc"], ["1efb2384f4547ca7337b22cd1223c61fc5e6c99784781b296f71a2a4ea2ff127"], ["3e1c07ead142d48f6da5f0d4a652d05a58306e2d2ea3b6b33784237bf8c3eaad"], ["32d68325e545b0ac0346c0331a3432ce82a

**CP6.7 — La distance de Fréchet de deux gaussiennes du plan de covariances proportionnelles, et le biais de la distance entre deux moyennes estimées**

In [9]:
wbm.record("CP6.7a", floats(LAM_AB_7), decimals=6, mistakes={**same_message('la question porte sur Σ_A Σ_B, pas sur Σ_A seule', as_typed(EIG_A_7, LAM_AB_7)), **same_message('la question demande les valeurs propres de Σ_A Σ_B, avant la racine', as_typed(ROOTS_7, LAM_AB_7))})
wbm.record("CP6.7b", COV_7, decimals=6, mistakes={'le terme croisé est 2 Σᵢ √λᵢ': as_typed(COV_ONCE_7, COV_7), 'le terme croisé porte sur les racines des valeurs propres': as_typed(COV_NO_ROOT_7, COV_7), 'les traces des covariances ne suffisent pas : retranche le terme croisé': as_typed(TRACES_7, COV_7)})
wbm.record("CP6.7c", W2_7, decimals=4, mistakes={'la distance de Fréchet est W₂, la racine de la formule': W2_7 ** 2, 'la distance de Fréchet tient compte des moyennes et des covariances': math.sqrt(COV_7), 'la part des moyennes est le carré de la distance entre elles': GAP_NORM_7})
wbm.record("CP6.7d", BIAS_7, decimals=6, mistakes={'une moyenne de n relevés a pour covariance Σ/n': as_typed(NO_N_7, BIAS_7), 'chaque trace se divise par la taille de son propre échantillon': as_typed(SWAP_N_7, BIAS_7), "l'espérance contient aussi l'écart des vraies moyennes": as_typed(NO_MEANS_7, BIAS_7), 'chaque moyenne estimée a sa propre variance : tr Σ_A / n_A + tr Σ_B / n_B, et non la somme des traces divisée par le nombre total de jours': as_typed(pooled(NA_7, NB_7), BIAS_7)})
wbm.record("CP6.7e", SAME_7, decimals=6, mistakes={'les deux échantillons viennent de deux fermes différentes': as_typed(SAME_A_7, SAME_7), 'deux échantillons, deux moyennes estimées : deux termes tr Σ / n': set().union(*(as_typed(y, SAME_7) for y in ONE_TERM_7)), "l'espérance contient aussi l'écart des vraies moyennes": as_typed(NO_MEANS_E_7, SAME_7), 'chaque moyenne estimée a sa propre variance : tr Σ_A / n_A + tr Σ_B / n_B, et non la somme des traces divisée par le nombre total de jours': as_typed(pooled(NE_7, NE_7), SAME_7)})

WBM_ANSWER {"decimals": 6, "element_coarse": [["6ea925dbbcedcdfee167e9e2e40570de3de52fc8aeb01386230d7842607f6898"], ["6959bc81cd09fc33b6339c73b13ba393aab4ecdca9df8b048b6c31920eba454e"]], "element_coarse_any": [["a27c04f207a815b8711ef42f283908c2a580b198e69e42679e9b40b08eb4e4e9", "57bf714d56df83d76b83abb78e88b7213b4018f3e991d4c53d6613feb19fb3a9", "61475a19620520cf5eb55a0990c090c459181f69d2a4d4d361150e34c2329671", "0f4be47d77cf777f1b0078e6bda5839a2fd88c9d843ba5d1508270e8ed03dda4", "ed897faeb0e5a0395f2fefe595820d76064cccb3a03f08054c7f17ee6c13486f"], ["c229b0aa2c0e83b33592e6b10b2dbf0088f5f3ab189c466d3356cacc8dd6842f", "4a26bdfebb834177bef43107226f1ddf78b54f6a00658f694dc4de5dc05bdd39", "983d2a0cd6352835a3bcc608983ec394385e2c935345de23f7f91a72bc29fe95", "b6fb7ddcf820d81aed42ba3bb4c066f9016b90d3af9403dd742937e6ddeaa0b3", "43fb5c59c35deacf92e02cd57371a0daf8b01f422a8e1b0219e665f80eb62fca"]], "element_hashes": ["de3cd7eb680239b30ab081cc52eae4710494f0731fb3d4a74e4718322e83ad6e", "cd1cf7765d3d9e21a

**CP6.9 — Corriger une copie : cinq erreurs, une par chapitre (les lettres seulement)**

In [10]:
wbm.record("CP6.9a", WRONG_9, mistakes={'cinq lignes sont fausses : ta réponse en contient moins': FOUR_9, 'exactement cinq lignes sont fausses': SIX_9, 'au moins une des lignes que tu donnes est juste : relis-les une par une': FIVE_9, 'ce sont les lignes justes : la question demande les fausses': RIGHT_9})

WBM_ANSWER {"hash": "85f361d2763e00fda25c7b35f84b0d0a13d6e5e0fb3f5f833b148d7fa14ed1cb", "id": "CP6.9a", "kind": "str", "mistakes": {"02575c9f4a0ce498d4e13cdd3e6504220a6bcdea312e1ed6bf66e0c44511f439": "au moins une des lignes que tu donnes est juste : relis-les une par une", "04e87aa8b74ae9373f4aa216dd03871fca9e274086501cb080cf7e708f38169e": "exactement cinq lignes sont fausses", "08224884cad618a09605f1d989272dd80205a112435ed8494572f3f2dfcb2591": "cinq lignes sont fausses : ta réponse en contient moins", "09f3e3cba981cd6435403767a14fa075b513ea24e0ba877ba05400e092b3c569": "au moins une des lignes que tu donnes est juste : relis-les une par une", "0e2db7f747e823a638d63e2ec57ef9528fcfda5f6d0528418241bbe6e2465312": "exactement cinq lignes sont fausses", "0e3f100149f9ab291fab571c3e542118313b8478a1a2f00bc35bb1b937b84842": "cinq lignes sont fausses : ta réponse en contient moins", "14b0df33dfadedff9378942efcd1da77608061de8e5f3cddb10df571d570e389": "cinq lignes sont fausses : ta réponse en cont

**CP6.10 — Problème de synthèse : une file d'attente à trois états suivie de façon gloutonne, puis échantillonnée après la troncature top-p de ses transitions**

In [11]:
wbm.record("CP6.10a", GREEDY_10, mistakes={"le glouton lit la ligne de l'état présent et y prend l'issue la plus probable": COLUMN_10})
wbm.record("CP6.10b", [PRE_10, PERIOD_10], mistakes={'la période est la longueur du cycle, sans la queue': [PRE_10, PRE_10 + PERIOD_10], 'la prépériode compte les états du début qui ne reviennent jamais': [0, PERIOD_10]})
wbm.record("CP6.10c", KEPT1_10, mistakes={**same_message('le top-p prend les issues une à une jusqu\'à ce que leur masse atteigne m', [LONGER_10, SHORTER_10]), 'range les états gardés par probabilité décroissante': sorted(KEPT1_10)})
wbm.record("CP6.10d", floats(PS1_10), decimals=4, mistakes={'une loi tronquée est encore une loi : sa somme vaut 1': floats(KEPT_ONLY_10[1]), 'une loi conditionnelle garde les rapports entre les probabilités gardées': floats(SPREAD_10)})
wbm.record("CP6.10e", KL1_10, decimals=4, mistakes={"dans le coût d'une troncature, P(S) est la masse gardée, pas la masse retirée": -math.log(1 - MASS1_10), 'les divergences KL sont en nats : logarithme népérien': -math.log2(MASS1_10), "la KL d'une troncature n'est pas la masse retirée": float(1 - MASS1_10)})
wbm.record("CP6.10f", floats(P_TRUNC_10), decimals=4, mistakes={'chaque ligne de P′ est une loi : sa somme vaut 1': floats(KEPT_ONLY_10), "une ligne garde ses issues jusqu'à ce que leur masse atteigne m, pas seulement la plus probable": ONE_HOT_10})
wbm.record("CP6.10g", floats(PI_NEW_10), decimals=4, mistakes={'la loi demandée est celle de la chaîne tronquée': floats(PI_OLD_10), 'une loi stationnaire a pour somme 1': floats(UNNORMALISED_10), "à l'équilibre, le flux de 1 vers 2 égale le flux de 2 vers 1 : vérifie lequel multiplie π′₁": floats(SWAPPED_10)})

WBM_ANSWER {"decimals": 0, "element_hashes": ["64e771e022ad04090a918feb9bd1f4119fbab922eec6de1d1e9b929def3e555f", "6f1093fdc7d4fabfb550bceac1dd54f0f60ad55ceaa6b2488a0118e7fcf66112", "d0b5907659e8d14d1a41db3805f64283718c988a045df4dc04a38378caad5e46"], "hash": "ca941beb94c5c4051834fc5aa7cb8883183ba27e7571dab0adb61637905386bc", "id": "CP6.10a", "integer": true, "kind": "array", "mistakes": {"f6d77b9b6aa3d6d569361c44213bca9081b2e131bf5642547437f02f8473bd9b": "le glouton lit la ligne de l'état présent et y prend l'issue la plus probable"}, "shape": [3]}
📝 Ex CP6.10a : réponse enregistrée (array, 0 décimale).
WBM_ANSWER {"decimals": 0, "element_hashes": ["71a2c68b5dfdf68dd46f88a434982f10240615236ccf0e161c8cc66504f35cfd", "ada700d46dcbdd095659b8978ac2333fb463d481de15a1cef052c87bbe10a4bf"], "hash": "a81a5e2f99a87fbc209752f42a19b7ad7f2fcc22c20f1ae372749e2aeaa15d08", "id": "CP6.10b", "integer": true, "kind": "array", "mistakes": {"6fa8ef61b825fe4aac54b57340281bfaa56da67e9f40bc12556b074ebbedc63d"